#### Dependency Setup

In [2]:
import os
import numpy as np
import pandas as pd
import pyomo.environ as pyo

os.environ['NEOS_EMAIL'] = 'kristofers-franks.petersons@tum.de'
FILENAME = 'Stage0.csv'

In [11]:
# Load data from CSV
data = pd.read_csv(FILENAME, sep=';')
a1 = data['a1'].values
a2 = data['a2'].values
c = data['c'].values
n = len(data)

# Define the model
model = pyo.ConcreteModel()

# Define the set of variables (indices)
model.I = pyo.Set(initialize=range(n))

# Define optimization variables
model.x = pyo.Var(model.I, domain=pyo.Reals)

# Define constraints: x_i >= c_i
def constraint_rule(model, i):
    return model.x[i] >= c[i]
model.constraints = pyo.Constraint(model.I, rule=constraint_rule)

# Define objective function: minimize sum(a2[i]*x[i]^2 + a1[i]*x[i])
def objective_rule(model):
    return sum(a2[i] * model.x[i]**2 + a1[i] * model.x[i] for i in model.I)

model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

# Solve the model using NEOS
solver = pyo.SolverManagerFactory('neos')
results = solver.solve(model, opt='cplex')

# Display results
print("Optimization Status:", results.solver.status)
print("Termination Condition:", results.solver.termination_condition)
print("\nOptimal Solution:")
for i in model.I:
    print(f"x[{i}] = {pyo.value(model.x[i]):.6f}")
print(f"\nOptimal Objective Value: {pyo.value(model.obj):.6f}")

Optimization Status: ok
Termination Condition: optimal

Optimal Solution:
x[0] = 0.000000
x[1] = 1.000000
x[2] = 0.000000
x[3] = -0.750000
x[4] = 0.000000
x[5] = -2.000000
x[6] = -6.500000

Optimal Objective Value: -49.375000


In [12]:
# Verify the solution by checking constraint satisfaction
print("Constraint Verification (x_i >= c_i):")
for i in model.I:
    x_val = pyo.value(model.x[i])
    print(f"x[{i}] = {x_val:.6f} >= c[{i}] = {c[i]}: {x_val >= c[i] - 1e-6}")

# Manual calculation of objective function value to verify
manual_obj = sum(a2[i] * (pyo.value(model.x[i])**2) + a1[i] * pyo.value(model.x[i]) for i in range(n))
print(f"\nManual Objective Calculation: {manual_obj:.6f}")
print(f"Model Objective Value: {pyo.value(model.obj):.6f}")
print(f"Match: {abs(manual_obj - pyo.value(model.obj)) < 1e-6}")

Constraint Verification (x_i >= c_i):
x[0] = 0.000000 >= c[0] = 0: True
x[1] = 1.000000 >= c[1] = 1: True
x[2] = 0.000000 >= c[2] = 0: True
x[3] = -0.750000 >= c[3] = -5: True
x[4] = 0.000000 >= c[4] = 0: True
x[5] = -2.000000 >= c[5] = -2: True
x[6] = -6.500000 >= c[6] = -10: True

Manual Objective Calculation: -49.375000
Model Objective Value: -49.375000
Match: True
